# Análisis de regresión — Monóxido de carbono (CO) en el estado de Nueva York, 2022

**Curso:** Proyectos de Ingeniería 1 — Guía 1: Regresión
**Autor:** Ruben Andre Cabrera Cermeño — Universidad Peruana Cayetano Heredia
**Fuente de datos:** US EPA, *Outdoor Air Quality Data — Download Daily Data*
**Indicador:** `Daily Max 8-hour CO Concentration` (ppm)

## 1. Configuración e importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 120)

## 2. Carga de datos

El archivo `Data1_CO_New_York.csv` debe estar disponible en el entorno. La celda
siguiente lo busca en la raíz y en la carpeta `datos/`; si no lo encuentra y se
está ejecutando en Google Colab, abre automáticamente el diálogo de subida.

In [ ]:
import os
import glob

NOMBRE = "Data1_CO_New_York.csv"
CONC   = "Daily Max 8-hour CO Concentration"

def localizar_csv(nombre):
    """Busca el CSV en las rutas habituales; si no existe, solicita la subida en Colab."""
    candidatos = [nombre, f"datos/{nombre}", f"data/{nombre}", f"/content/{nombre}"]
    for ruta in candidatos:
        if os.path.exists(ruta):
            return ruta

    # Como respaldo, cualquier CSV de la EPA presente en el directorio
    sueltos = glob.glob("*.csv") + glob.glob("datos/*.csv") + glob.glob("/content/*.csv")
    sueltos = [s for s in sueltos if "sample_data" not in s]
    if sueltos:
        print(f"No se encontró '{nombre}'. Se usará: {sueltos[0]}")
        return sueltos[0]

    # En Colab, pedir el archivo al usuario
    try:
        from google.colab import files
        print(f"Sube el archivo '{nombre}':")
        subidos = files.upload()
        return list(subidos.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"No se encontró '{nombre}'. Colócalo junto al notebook o en la carpeta 'datos/'."
        )

RUTA = localizar_csv(NOMBRE)
print("Archivo cargado desde:", RUTA)

df = pd.read_csv(RUTA)
df.columns = df.columns.str.strip()

# Validación de que el archivo es el esperado
assert CONC in df.columns, f"El archivo no contiene la columna '{CONC}'. Columnas: {list(df.columns)}"

print("Dimensiones:", df.shape)
print("Columnas:", list(df.columns))
df.head()

In [ ]:
df.info()

In [ ]:
df[[CONC, "Daily AQI Value", "Daily Obs Count", "Percent Complete"]].describe()

## 3. Preprocesamiento

1. Conversión de la columna `Date` al tipo fecha.
2. Eliminación de registros con fecha o concentración nula.
3. Revisión de la cobertura temporal de cada estación de monitoreo.
4. Construcción de una serie diaria única promediando las estaciones.

In [ ]:
df["Fecha"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
df = df.dropna(subset=["Fecha", CONC])

print("Años presentes:", sorted(df["Fecha"].dt.year.unique()))
print("Unidades:", df["Units"].unique())
print("\nCobertura por estación:")
df.groupby("Local Site Name")["Fecha"].agg(["min", "max", "count"])

In [ ]:
print("Registros por condado:")
print(df["County"].value_counts())

In [ ]:
serie = (df.groupby("Fecha")[[CONC, "Daily AQI Value"]]
           .mean()
           .rename(columns={CONC: "CO", "Daily AQI Value": "AQI"})
           .reset_index()
           .sort_values("Fecha")
           .reset_index(drop=True))

# Variables explicativas
serie["t"]     = (serie["Fecha"] - serie["Fecha"].min()).dt.days   # días transcurridos
serie["mes"]   = serie["Fecha"].dt.month
serie["doy"]   = serie["Fecha"].dt.dayofyear
serie["dow"]   = serie["Fecha"].dt.dayofweek                        # 0 = lunes
serie["finde"] = (serie["dow"] >= 5).astype(int)                    # 1 = sábado o domingo

print("Serie diaria:", serie.shape)
serie.head()

## 4. Análisis exploratorio

In [ ]:
serie[["CO", "AQI"]].describe().round(4)

In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(x="mes", y="CO", data=serie, color="#4c72b0")
plt.title("Figura 1. Estacionalidad mensual del CO — Nueva York, 2022")
plt.xlabel("Mes"); plt.ylabel("CO (ppm)")
plt.tight_layout(); plt.savefig("fig1_estacionalidad.png", bbox_inches="tight"); plt.show()

serie.groupby("mes")["CO"].agg(["mean", "median", "std"]).round(4)

In [ ]:
# Efecto de día de semana (sin gráfico: la diferencia no es visible sin controlar por otras variables)
print(serie.groupby("finde")["CO"].agg(["mean", "std", "count"]).round(4))

t_stat, p_val = stats.ttest_ind(serie.loc[serie["finde"] == 0, "CO"],
                                serie.loc[serie["finde"] == 1, "CO"],
                                equal_var=False)
print(f"\nPrueba t (días hábiles vs fin de semana): t = {t_stat:.3f}, p = {p_val:.4f}")
print("\nMedia por día de la semana (0 = lunes):")
print(serie.groupby("dow")["CO"].mean().round(4))

In [ ]:
por_sitio = (df.groupby("Local Site Name")[CONC]
               .agg(["mean", "max", "count"])
               .sort_values("mean", ascending=False)
               .round(3))

plt.figure(figsize=(8, 4.5))
por_sitio["mean"].sort_values().plot(kind="barh", color="#c44e52")
plt.title("Figura 2. CO promedio anual por estación de monitoreo")
plt.xlabel("CO (ppm)"); plt.ylabel("")
plt.tight_layout(); plt.savefig("fig2_por_estacion.png", bbox_inches="tight"); plt.show()

por_sitio

In [ ]:
# Verificación de consistencia: el AQI es una transformación determinística de la concentración
print("Correlación de Pearson CO–AQI:", round(serie["CO"].corr(serie["AQI"]), 4))
print("\nNo se utiliza el AQI como variable explicativa por ser redundante con CO.")

## 5. Modelo M1 — Regresión lineal simple

Especificación: `CO = b0 + b1 * t + e`, donde `t` son los días transcurridos desde
el 1 de enero de 2022.

In [ ]:
X = serie[["t"]]
y = serie["CO"]

m1_sk = LinearRegression().fit(X, y)
pred1 = m1_sk.predict(X)

print(f"Intercepto : {m1_sk.intercept_:.4f} ppm")
print(f"Pendiente  : {m1_sk.coef_[0]:.6f} ppm/día  ->  {m1_sk.coef_[0]*365:.4f} ppm/año")
print(f"R²         : {r2_score(y, pred1):.4f}")
print(f"RMSE       : {np.sqrt(mean_squared_error(y, pred1)):.4f} ppm")
print(f"MAE        : {mean_absolute_error(y, pred1):.4f} ppm")

plt.figure(figsize=(12, 4))
plt.scatter(serie["Fecha"], y, s=10, alpha=.4, label="Observado")
plt.plot(serie["Fecha"], pred1, color="red", lw=2, label="Regresión lineal (M1)")
plt.title("Figura 3. Serie diaria de CO y recta de regresión del modelo M1")
plt.xlabel("Fecha"); plt.ylabel("CO (ppm)"); plt.legend()
plt.tight_layout(); plt.savefig("fig3_regresion_simple.png", bbox_inches="tight"); plt.show()

In [ ]:
m1 = sm.OLS(y, sm.add_constant(X)).fit()
print(m1.summary())

## 6. Modelo M2 — Regresión lineal múltiple

Se incorporan tres bloques de información adicionales:

- **Estacionalidad anual:** armónicos `sin(2*pi*d/365)` y `cos(2*pi*d/365)`.
- **Efecto de día de semana:** variable indicadora `finde` (1 = sábado o domingo),
  pertinente porque el CO proviene mayoritariamente del tráfico vehicular.
- **Persistencia:** `lag1`, la concentración del día anterior.

In [ ]:
serie["sin_doy"] = np.sin(2*np.pi*serie["doy"]/365)
serie["cos_doy"] = np.cos(2*np.pi*serie["doy"]/365)
serie["lag1"]    = serie["CO"].shift(1)

d = serie.dropna().reset_index(drop=True)
d["log_CO"]   = np.log(d["CO"])
d["log_lag1"] = np.log(d["lag1"])

feats2 = ["t", "sin_doy", "cos_doy", "finde", "lag1"]

m2 = sm.OLS(d["CO"], sm.add_constant(d[feats2])).fit()
print(m2.summary())
print("\nDurbin-Watson:", round(sm.stats.durbin_watson(m2.resid), 3))

### 6.1 Validación con partición temporal

La partición es cronológica, no aleatoria, para evitar que información posterior
se filtre al conjunto de entrenamiento:

- Entrenamiento: enero – septiembre de 2022
- Prueba: octubre – diciembre de 2022

In [ ]:
corte = d["Fecha"] < "2022-10-01"
y_te  = d.loc[~corte, "CO"]

def evaluar(feats, dep="CO", log=False, etiqueta=""):
    mod = LinearRegression().fit(d.loc[corte, feats], d.loc[corte, dep])
    p = mod.predict(d.loc[~corte, feats])
    if log:
        p = np.exp(p)
    r2, rmse, mae = (r2_score(y_te, p),
                     np.sqrt(mean_squared_error(y_te, p)),
                     mean_absolute_error(y_te, p))
    print(f"{etiqueta:<28} R² = {r2:7.4f} | RMSE = {rmse:.4f} | MAE = {mae:.4f}")
    return mod, p, (r2, rmse, mae)

print(f"Entrenamiento: {corte.sum()} días | Prueba: {(~corte).sum()} días\n")
_,  p1, res1 = evaluar(["t"], etiqueta="M1 (solo tendencia)")
m2_sk, p2, res2 = evaluar(feats2, etiqueta="M2 (múltiple, niveles)")

## 7. Modelo M4 — Especificación log-log parsimoniosa

En M2 varios coeficientes resultan no significativos y los residuos presentan
asimetría positiva. Se estima una versión logarítmica que conserva únicamente
los regresores significativos y mantiene la coherencia de escala rezagando
`ln(CO)` en lugar de `CO`:

`ln(CO) = b0 + b1*cos_doy + b2*finde + b3*ln(CO_{t-1}) + e`

In [ ]:
feats4 = ["cos_doy", "finde", "log_lag1"]

m4 = sm.OLS(d["log_CO"], sm.add_constant(d[feats4])).fit()
print(m4.summary())
print("\nDurbin-Watson:", round(sm.stats.durbin_watson(m4.resid), 3))

In [ ]:
m4_sk, p4, res4 = evaluar(feats4, dep="log_CO", log=True, etiqueta="M4 (log-log parsimonioso)")

plt.figure(figsize=(12, 4))
plt.plot(d.loc[~corte, "Fecha"], y_te, label="Real", lw=1.5)
plt.plot(d.loc[~corte, "Fecha"], p4, label="Predicho (M4)", lw=1.5)
plt.title("Figura 4. Modelo M4 — valores reales y predichos en el conjunto de prueba (oct–dic 2022)")
plt.xlabel("Fecha"); plt.ylabel("CO (ppm)"); plt.legend()
plt.tight_layout(); plt.savefig("fig4_m4_prueba.png", bbox_inches="tight"); plt.show()

## 8. Verificación de supuestos

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
sm.qqplot(m2.resid, line="45", fit=True, ax=ax[0]); ax[0].set_title("M2 (niveles)")
sm.qqplot(m4.resid, line="45", fit=True, ax=ax[1]); ax[1].set_title("M4 (log-log)")
fig.suptitle("Figura 5. Normalidad de los residuos: comparación de especificaciones", y=1.02)
plt.tight_layout(); plt.savefig("fig5_qq.png", bbox_inches="tight"); plt.show()

for nombre, mod in [("M2 (niveles)", m2), ("M4 (log-log)", m4)]:
    jb, jb_p = sm.stats.jarque_bera(mod.resid)[:2]
    print(f"{nombre:<14} asimetría = {stats.skew(mod.resid):6.3f} | "
          f"curtosis = {stats.kurtosis(mod.resid):6.3f} | "
          f"Jarque-Bera = {jb:7.3f} (p = {jb_p:.4f})")

## 9. Tabla comparativa de modelos

In [ ]:
resumen = pd.DataFrame({
    "Modelo": ["M1 — Simple (t)", "M2 — Múltiple (niveles)", "M4 — Log-log parsimonioso"],
    "R² ajuste":  [m1.rsquared, m2.rsquared, m4.rsquared],
    "R² ajustado":[m1.rsquared_adj, m2.rsquared_adj, m4.rsquared_adj],
    "F":          [m1.fvalue, m2.fvalue, m4.fvalue],
    "p (F)":      [m1.f_pvalue, m2.f_pvalue, m4.f_pvalue],
    "Durbin-Watson": [sm.stats.durbin_watson(m1.resid),
                      sm.stats.durbin_watson(m2.resid),
                      sm.stats.durbin_watson(m4.resid)],
    "R² prueba":  [res1[0], res2[0], res4[0]],
    "RMSE prueba":[res1[1], res2[1], res4[1]],
    "MAE prueba": [res1[2], res2[2], res4[2]],
})
resumen.round(4)

## 10. Conclusiones

1. La concentración diaria máxima de 8 horas de CO en Nueva York durante 2022
   promedió 0.2974 ppm, muy por debajo del estándar nacional de calidad del aire
   de 9 ppm, con un máximo de 0.75 ppm.
2. El modelo de tendencia simple no es significativo (p = 0.099) y en validación
   produce un R² negativo, es decir, predice peor que la media del periodo. Con
   un solo año de datos la tendencia lineal no es separable de la estacionalidad.
3. El modelo múltiple alcanza un R² de 0.393 y el log-log parsimonioso un 0.409,
   con todos sus coeficientes significativos y el mejor desempeño en validación
   (RMSE = 0.1186 ppm frente a 0.1967 del modelo simple).
4. La persistencia diaria es el predictor dominante: la elasticidad estimada de
   0.532 indica que un aumento del 1 % en el CO del día previo se asocia a un
   aumento del 0.53 % en el día actual.
5. Se confirma el patrón estacional esperable para un contaminante de origen
   vehicular: concentraciones más altas en los meses fríos (octubre a febrero) y
   mínimas en agosto.
6. El efecto de fin de semana no es detectable al comparar medias simples
   (prueba t: p = 0.359), pero resulta significativo al controlar por
   estacionalidad y persistencia en el modelo M4 (−5.0 %, p = 0.042). Es un
   ejemplo de cómo un efecto real puede quedar oculto tras la variabilidad
   diaria si no se especifica el modelo adecuadamente.
7. El gradiente espacial es marcado: la estación Queens College Near Road
   (0.434 ppm) triplica el promedio de Pinnacle State Park (0.125 ppm), estación
   rural de fondo, lo que respalda la atribución del CO al tráfico vehicular.